In [ ]:
# Install required libraries
!pip install langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers transformers accelerate bitsandbytes chromadb

In [ ]:
# -----------------------
# 1. Imports
# -----------------------
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
# -----------------------
# 2. Load Documents
# -----------------------
# Example: create a sample text file
with open("sample.txt", "w") as f:
    f.write("""LangChain makes it easy to build applications with LLMs.
    Retrieval-Augmented Generation (RAG) is a powerful technique that
    combines retrieval and generation to improve factual accuracy.""")

loader = TextLoader("sample.txt")
docs = loader.load()

In [ ]:
# -----------------------
# 3. Split Text into Chunks
# -----------------------
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(docs)


In [ ]:
# -----------------------
# 4. Embeddings + Chroma Vector DB
# -----------------------
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

In [ ]:
# Create Chroma database in-memory (no persistence)
vectorstore = Chroma.from_documents(documents, embeddings)

In [ ]:
# -----------------------
# 5. Load Open-Source LLM (no API key)
# -----------------------

model_name = "tiiuae/falcon-7b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True 
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.2,
    repetition_penalty=1.1
)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
# -----------------------
# 6. RAG Pipeline
# -----------------------
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    chain_type="stuff"
)

In [ ]:
# -----------------------
# 7. Ask Questions
# -----------------------
query = "What is Retrieval-Augmented Generation?"
result = qa_chain.run(query)

print("Q:", query)
print("A:", result)